# Reading the output files

[`02_quickstart.ipynb`](02_quickstart.ipynb) ran the pipeline and plotted the result. A run writes three kinds of file, and this notebook opens each one:

- the **image file**, the decoded particle images;
- the **proc file**, one row per particle: its size, shape, habit and whether it was rejected;
- the **sd file**, the size distribution, one row per second and one column per size bin;
- and beside each, a one-line **`.call` file** that records how that file was made.

It reads them with `netCDF4` where the file's own layout (dimensions, types, attributes) is the point, and with `xarray` where only the values are. No campaign data: it builds its own demo flight.

**Before this:** [05_triage_and_gallery](05_triage_and_gallery.ipynb) (Looking at particles: a contact sheet, a gallery, and the triage page). **Next:** [07_working_with_an_ai_assistant](07_working_with_an_ai_assistant.ipynb). **Needs:** `pip install "arrayprobe[zarr]"` (xarray).

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

## 1. Build and process a demo flight


In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/1_getting_started/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

import sys
import tempfile
from pathlib import Path

sys.path.insert(0, "scripts")
import make_demo_flight  # noqa: E402
import run_flight  # noqa: E402

run_flight.log = lambda *a, **kw: None  # quiet the tool's own progress prints

tmp = tempfile.TemporaryDirectory()
out = Path(tmp.name)
# The public package runs the v3.5 engine: this sets engine, profile and a constant-TAS aircraft file.
run_flight.use_engine_35(out)
raw = out / "base160101000000.2DS"
make_demo_flight.build_demo_flight(raw, n_seconds=5)

image = run_flight.stage2(raw, out)
proc = run_flight.stage3(image, out)
sd = run_flight.stage4(proc, raw.stem, out)
print(f"proc: {proc.name}\nsd:   {sd.name}")


proc: proc.base160101000000.H.nc
sd:   sd.base160101000000.H.nc


## 2. The proc file's variable contract

Every particle in the flight gets one row across all of these variables -- one
netCDF record per particle, not per second. `holroyd_habit` and `image_auto_reject` hold
the ASCII code of a single character, `int16` in stock UIOOPS (`imgProc_sm.m:923`) and in this
demo's stock v3.5 run (the cell below prints the dtypes). Under stock v3.5 the hollow test
runs, so a hollow particle is coded `'h'` and kept. `chr(int(value))` recovers the
letter, matching `docs/11-glossary.md`'s habit/reject-code tables.


In [2]:
import netCDF4

with netCDF4.Dataset(str(proc)) as ds:
    n_particles = ds.dimensions["time"].size
    print(f"{n_particles} particle(s), {len(ds.variables)} variables\n")
    for name in ("image_length", "image_width", "image_area",
                 "holroyd_habit", "image_auto_reject", "part_z"):
        v = ds.variables[name]
        print(f"{name:16s} {str(v.dtype):8s} {v.dimensions}")
    print()
    habit = ds.variables["holroyd_habit"][0]
    reject = ds.variables["image_auto_reject"][0]
    print(f"particle 0: habit={chr(int(habit))!r}, image_auto_reject={chr(int(reject))!r}")


5 particle(s), 40 variables

image_length     int16    ('time',)
image_width      int16    ('time',)
image_area       float64  ('time',)
holroyd_habit    int16    ('time',)
image_auto_reject int16    ('time',)
part_z           float64  ('time',)

particle 0: habit='s', image_auto_reject='h'


## 3. The sd file and the used-bin convention

`bin_mid` names every diameter bin the size distribution is written over -- all 128
of them, 1-indexed in every UIOOPS document and MATLAB comment. The first five (bins
1-5, Python indices `0:5`) sit at the 2D-S's diode-pitch noise floor and are excluded
from every headline this project publishes; **used bins are 6-128**, `bin_mid[5:]`.
See `docs/11-glossary.md` for where this convention is fixed once.


In [3]:
import numpy as np
import xarray as xr

# Raw stored values (mask_and_scale=False), exactly as the file holds them.
with xr.open_dataset(sd, mask_and_scale=False, decode_times=False) as ds:
    bin_mid = ds["bin_mid"].values
    conc = ds["conc_minR"].values  # (time, bin)

print(f"bin_mid: {bin_mid.shape[0]} bins, {bin_mid[0]:.4g} to {bin_mid[-1]:.4g} mm")
print(f"conc_minR: {conc.shape} (time, bin)")

all_bins = np.nan_to_num(conc).sum()
used_bins = np.nan_to_num(conc[:, 5:]).sum()
print(f"summed concentration, all 128 bins:  {all_bins:.6g}")
print(f"summed concentration, used bins 6-128: {used_bins:.6g}")

# The unit string the file stores for the concentration.
with xr.open_dataset(sd, decode_times=False) as ds_u:
    attrs = ds_u["conc_minR"].attrs
    print("units of conc_minR:", attrs.get("units", attrs.get("Units")))


bin_mid: 128 bins, 0.01 to 1.28 mm
conc_minR: (4, 128) (time, bin)
summed concentration, all 128 bins:  1.87434
summed concentration, used bins 6-128: 0
units of conc_minR: cm-4


## 4. The `.call` sidecar

Reproducing an archived product requires knowing which call produced it, and no
UIOOPS output file records that on its own -- so `run_flight.py` writes one alongside
every output it makes. It is plain text, one line, never inside the netCDF file
itself (an attribute of the port's own would break byte-fidelity with MATLAB).


In [4]:
from arrayprobe_uiops import provenance

call_sidecar = sd.with_name(sd.name + ".call")
stamp = provenance.parse(call_sidecar.read_text())   # the package's own stamp reader
for key, value in stamp.fields.items():
    if value.startswith("(") and value.endswith(")"):    # a group of settings: one per line
        print(f"{key}:")
        for item in value[1:-1].split(","):
            sub_key, _, sub_value = item.partition("=")
            print(f"    {sub_key:<24s} {sub_value}")
    else:
        print(f"{key:<28s} {value}")


engine                       3.5
iat_file                     2DS_intArrThreshold_MISSING.cdf
profile                      stock-v3.5
reject:
    rej_zero_area            True
    rej_split                True
    rej_hollow               True
    shadowed_area            25
    max_edge_img_ar          6
    max_comp_img_ar          5
shadow_diode                 none
tas_file                     aircraft.cdf
call:
    iRectEllipse             0
    iCalcAllDiodeStats       0
    iQC                      0
    iNewHabit                0
    tas_ratio                off
software                     UIOOPS_v3.5
stray_pixel                  off
sizedist:
    iaThreshType             1
    iCreateAspectRatio       0
    iCreateBad               0
    iSaveIntArrSV            0
bins                         2ds-ladder(V11-fix)
time_sv_fix                  on
zero_sv_seconds              0
time_grid_fix                0
time_grid_unresolved         0
plausibility:
    sec                    

## What to try next

- [`7_analysis_and_outputs/08_enhancements_on_a_product.ipynb`](../7_analysis_and_outputs/08_enhancements_on_a_product.ipynb) -- the optional enhancements run on a product like this one: moments, the two-probe check, CF export and more.
- [`01_validate_against_a_product.ipynb`](../3_validation/01_validate_against_a_product.ipynb) -- comparing a port output against an
  archived reference product, both denominators, how a residual is classified.
- `docs/11-glossary.md` -- every convention fixed once: used bins,
  both-present vs. non-zero denominators, habit and reject-code letters.
